In [1]:
import requests
import re
import numpy as np
from bs4 import BeautifulSoup
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
url = 'https://www.eazydiner.com/'

In [3]:
Request = requests.get(url)

In [4]:
Request

<Response [200]>

In [5]:
# Lists to store data
Restaurant_Name = []
Rating = []
Area = []
Dishes = []
Amount = []
Discount = []

# List of all cities 
cities = ['jaipur', 'goa', 'patna', 'mumbai', 'bengaluru']

# Scraping loop
for city in cities:
    for page_num in range(1, 16):  # Adjust number of pages if needed
        url = f"https://www.eazydiner.com/restaurants?location={city}&page={page_num}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Restaurant titles
        titles = soup.find_all('a', class_='ellipsis listing_res_name__uVIN8')
        for t in titles:
            Restaurant_Name.append(t.text.strip())
        
        # Ratings
        ratings = soup.find_all('tspan')
        for i in ratings:
            j = i.text
            if j != '%':
                if j:
                    Rating.append(j)
                else:
                    Rating.append(np.nan)
        
    
        
        # Area / address
        areas = soup.find_all('div', class_='font-12 ellipsis flex listing_res_address__LuVOm')
        for a in areas:
            Area.append(a.text.strip())
        
        # Dishes / cuisines
        cuisines = soup.find_all('div', class_='flex align-v-center ellipsis listing_multi_cuisines__zLLXK')
        for d in cuisines:
            Dishes.append(d.text.strip() if d else np.nan)
        
        # Approx cost
        approx_txt = []
        approx = soup.find_all('div', class_='flex align-v-center')
        for a in approx:
            approx_txt.append(a.text.strip())
        for i in approx_txt:
            if re.match(r"₹ +\d+ [A-Za-z].+", i):
                Amount.append(i)
        
        # Discount offers
        discounts = soup.find_all('div', class_='ellipsis listing_offer_text__LXKw1')
        for d in discounts:
            Discount.append(d.text.strip())

# Print lengths of all lists
print("Restaurant_Name:", len(Restaurant_Name))
print("Rating:", len(Rating))
print("Area:", len(Area))
print("Dishes:", len(Dishes))
print("Amount:", len(Amount))
print("Discount:", len(Discount))


Restaurant_Name: 675
Rating: 675
Area: 675
Dishes: 675
Amount: 675
Discount: 675


In [6]:
data = pd.DataFrame({
    "Restaurant_Name": Restaurant_Name,
    "Rating": Rating,
    "Area": Area,
    "Dishes": Dishes,
    "Amount": Amount,
    "Discount": Discount
})

In [7]:
data

,Restaurant_Name,Rating,Area,Dishes,Amount,Discount
0,Brot Company,5.0,"Civil Lines, Jaipur","Bakery, Pan Asian, Continental",₹ 1600 for two approx.,20% Off + 25% Off
1,Govindam Retreat,4.4,"Kanwar Nagar, Jaipur",Multicuisine,₹ 700 for two approx.,25% Off
2,Spice Court,4.4,"Civil Lines, Jaipur","Multicuisine, Beverages",₹ 800 for two approx.,10% Off + 25% Off
3,Cafe Auberge,4.8,"Civil Lines, Jaipur",Multicuisine,₹ 800 for two approx.,20% Off + 25% Off
4,Dyore Experiences,3.3,"C Scheme, Jaipur",Multicuisine,₹ 1800 for two approx.,20% Off + 25% Off
...,...,...,...,...,...,...
670,Kenzai - Asian Kitchen And Lounge,4.6,"Brigade Road, Central Bengaluru",Asian,₹ 2000 for two approx.,25% Off + 25% Off
671,Cahoots Classic Bar House,5.0,"Brigade Road, Central Bengaluru",Multicuisine,₹ 4000 for two approx.,10% Off + 25% Off
672,i-Bar,3.8,"The Park, Bengaluru",Cocktail Menu,₹ 2000 for two approx.,30% Off + 25% Off
673,Sea Rock Family Restaurant & Bar,4.7,"Seshadripuram, Central Bengaluru",Multicuisine,₹ 2000 for two approx.,10% Off + 25% Off


In [8]:
data.to_csv('eazydiner_dataset.csv', index=False)